# Type Hints & Best Practices

## What's covered

Two adjacent topics: how to add type information to Python code so that tools can check it, and the broader practices that keep a Python codebase healthy.

**Type hints:**

- The promise — types are documentation that tools can verify
- Types are NOT enforced at runtime — they're checked by `mypy`, `pyright`, `ruff`
- The simple cases — primitives, collections, callables, optionals
- The pre-3.9 import dance — `from typing import List, Dict, Optional` — and the modern built-in form
- `Optional[T]` and `T | None` (3.10+) — and why `None` is opt-in
- Union types — `int | str`
- `Literal[...]` — typing the exact set of allowed values
- `TypedDict` — typed dictionaries with known keys
- `Protocol` — structural typing, Python's answer to "duck typing with a type"
- Generic functions and classes — `TypeVar`, the 3.12 `def f[T](x: T) -> T:` syntax
- `Any`, `cast`, `# type: ignore` — escape hatches
- `mypy` / `pyright` — checking the code

**Best practices:**

- PEP 8 — the style guide, and the tools that automate it (`ruff`, `black`)
- Naming conventions
- Import sorting and grouping
- Project layout — `src/` layout, `pyproject.toml`, where tests go
- The `__init__.py` patterns
- Logging — `logging` module, levels, structured logs
- `print()` vs `logging` — when each fits
- Configuration via environment variables, with `python-dotenv` for development
- Performance — when to profile, what to measure, the `cProfile` and `timeit` starting points
- Style choices that age well — small functions, explicit data, hard-to-misuse APIs

## Part 1 — Type hints

## The promise — and what they don't do

Type hints (since Python 3.5) let you annotate parameters and return values with the types they expect:

```python
def greet(name: str, times: int = 1) -> str:
    return ("hello, " + name + ". ") * times
```

The promise:

- **Tools** (`mypy`, `pyright`, your IDE) check your code statically and flag mismatches before you run.
- **Humans** read the signature and know what to pass.
- **Refactors** become safer — rename a class and the type checker shows every site that needs updating.

What hints **don't** do: **they don't enforce anything at runtime.** Python ignores them when executing your code. Calling `greet(42, "no")` runs without error until the string operations in the body fail. If you want runtime validation, reach for `pydantic` or write `if not isinstance(...)` checks yourself.

The mental model: type hints are like comments that the type checker reads. The interpreter doesn't.

In [ ]:
def greet(name: str, times: int = 1) -> str:
    return ("hello, " + name + ". ") * times

print(greet("ganesh"))
print(greet("alice", 3))

# Runtime DOESN'T enforce the hints — this runs (and fails inside the body)
try:
    greet(42, "no")
except TypeError as e:
    print(f"runtime error: {e}")
# A type checker like mypy would have flagged the call BEFORE you ran

## The basics — primitives, collections, callables

Same-named types you already know: `int`, `float`, `str`, `bool`, `bytes`, `None`. For containers, since Python 3.9 you can use the built-in types directly as generics:

```python
def total(prices: list[float]) -> float: ...
def lookup(table: dict[str, int], key: str) -> int: ...
def first(pairs: list[tuple[str, int]]) -> tuple[str, int]: ...
```

Older code uses imports from `typing`:

```python
from typing import List, Dict, Tuple
def total(prices: List[float]) -> float: ...
```

Both forms work in 3.9+. New code should prefer the built-in lowercase form.

For callables, use `Callable[[Arg1, Arg2], Return]` from `collections.abc` (or `typing.Callable` pre-3.9):

```python
from collections.abc import Callable
def apply(f: Callable[[int], int], x: int) -> int:
    return f(x)
```

In [ ]:
from collections.abc import Callable

def total(prices: list[float]) -> float:
    return sum(prices)

def lookup(table: dict[str, int], key: str, default: int = 0) -> int:
    return table.get(key, default)

def first_pair(pairs: list[tuple[str, int]]) -> tuple[str, int]:
    return pairs[0]

def apply_n(f: Callable[[int], int], x: int, n: int) -> int:
    for _ in range(n):
        x = f(x)
    return x

print(total([1.5, 2.5, 3.0]))
print(lookup({"a": 1}, "a"))
print(first_pair([("first", 1), ("second", 2)]))
print(apply_n(lambda n: n * 2, 1, 5))   # 32

## `Optional[T]`, `T | None`, and union types

A value that could be `None` is **optional**. The classic form is `Optional[T]` from `typing`. Since 3.10, the cleaner form is `T | None`.

Same idea for unions: `Union[int, str]` (old) or `int | str` (3.10+). The new form is preferred in modern code.

A method without `-> None` and without a `return` returns `None` implicitly — but the type checker won't infer that. Write `-> None` explicitly on procedures.

In [ ]:
# Modern union syntax (3.10+)
def find_user(name: str) -> dict | None:
    db = {"alice": {"age": 30}, "bob": {"age": 25}}
    return db.get(name)

print(find_user("alice"))
print(find_user("carol"))

# Mixed types
def stringify(x: int | float | str) -> str:
    return str(x)

print(stringify(42))
print(stringify(3.14))
print(stringify("hi"))

# Older imports for the same idea
from typing import Optional, Union
def find_user_old(name: str) -> Optional[dict]: ...
def stringify_old(x: Union[int, float, str]) -> str: ...
# Both still valid; prefer the | form in new code.

## `Literal[...]` — exact set of allowed values

When a parameter accepts only a few specific *values*, `Literal[...]` documents the choices and lets the type checker enforce them.

```python
from typing import Literal

def sort_by(direction: Literal["asc", "desc"]) -> None: ...

sort_by("asc")    # ok
sort_by("up")     # mypy: Argument 1 must be "asc" or "desc"
```

This is Python's cleanest answer to "enum-but-just-three-strings." For larger sets, reach for a real `Enum` or a `Literal` of more values.

In [ ]:
from typing import Literal

def fetch(method: Literal["GET", "POST", "PUT", "DELETE"], url: str) -> str:
    return f"{method} {url}"

print(fetch("GET", "/users"))
print(fetch("POST", "/users"))
# fetch("HEAD", "/users")   # mypy: "HEAD" doesn't match any literal

## `TypedDict` — typed dictionaries

For a dict with known string keys (config blobs, JSON payloads), `TypedDict` documents the shape:

```python
from typing import TypedDict

class UserDict(TypedDict):
    name: str
    age: int
    active: bool

u: UserDict = {"name": "ganesh", "age": 30, "active": True}
```

The type checker now ensures every required key is present and that each value has the right type. Useful when you can't (or won't) replace the dict with a `dataclass`.

Mark optional keys with `total=False` or per-key `NotRequired[...]` (3.11+).

In [ ]:
from typing import TypedDict, NotRequired

class UserDict(TypedDict):
    name: str
    age: int
    email: NotRequired[str]   # optional (3.11+)

def describe(u: UserDict) -> str:
    base = f"{u['name']}, {u['age']}"
    if "email" in u:
        base += f" <{u['email']}>"
    return base

print(describe({"name": "ganesh", "age": 30}))
print(describe({"name": "alice", "age": 25, "email": "alice@example.com"}))

## `Protocol` — structural typing

A `Protocol` is Python's answer to "any class with these methods counts as one of these." It is **structural** — no inheritance needed; the type checker accepts any object whose shape matches.

This is the typed version of duck typing. `Iterable`, `Sized`, `Iterator`, `Sequence`, `Mapping` in `collections.abc` are all protocols.

```python
from typing import Protocol

class Greetable(Protocol):
    def greet(self) -> str: ...

def announce(thing: Greetable) -> None:
    print(thing.greet())

class Cat:               # NOT a subclass of Greetable
    def greet(self):
        return "meow"

announce(Cat())          # OK — Cat structurally satisfies Greetable
```

Use protocols when you want to type "anything with these methods" without forcing inheritance. They are the right modern answer instead of `ABC` for most "interface" cases.

In [ ]:
from typing import Protocol

class Sized(Protocol):
    def __len__(self) -> int: ...

class Hashable(Protocol):
    def __hash__(self) -> int: ...

def describe_size(thing: Sized) -> str:
    return f"size = {len(thing)}"

# Any object with __len__ works — no inheritance needed
print(describe_size([1, 2, 3]))
print(describe_size("hello"))
print(describe_size({"a": 1, "b": 2}))

# A custom class — works as long as it has __len__
class Inventory:
    def __init__(self, items):
        self.items = items
    def __len__(self):
        return len(self.items)

print(describe_size(Inventory([1, 2, 3, 4])))

## Generic functions and classes

When you want code that works for *any* type but with the type checker tracking it, you use a **TypeVar**.

Two syntaxes:

- **Pre-3.12** — `T = TypeVar("T")` then `def f(x: T) -> T:`
- **3.12+** — `def f[T](x: T) -> T:` — type parameter syntax baked into the language

The new form mirrors what most other languages (Java, Scala, TypeScript, Rust) have always had. Use it in 3.12+ code.

In [ ]:
# Old form — TypeVar
from typing import TypeVar

T = TypeVar("T")

def first(items: list[T]) -> T:
    return items[0]

print(first([1, 2, 3]))         # type checker knows result is int
print(first(["a", "b"]))        # type checker knows result is str

# New form (3.12+) — type parameter syntax
def last[T](items: list[T]) -> T:
    return items[-1]

print(last([1, 2, 3]))
print(last(["a", "b"]))

# Generic class — bound type parameter
class Stack[T]:
    def __init__(self):
        self._items: list[T] = []
    def push(self, x: T) -> None:
        self._items.append(x)
    def pop(self) -> T:
        return self._items.pop()
    def __len__(self) -> int:
        return len(self._items)

s: Stack[int] = Stack()
s.push(1); s.push(2); s.push(3)
print(s.pop())   # 3
print(len(s))

## `Any`, `cast`, and `# type: ignore` — escape hatches

Three ways to opt out of type checking when you have to:

- **`Any`** — the type that matches everything. Disables checking for that value. Use sparingly; an `Any` "infects" anything it touches.
- **`cast(T, x)`** — tells the type checker "treat `x` as type `T`," with no runtime conversion. Use when you know more than the checker does.
- **`# type: ignore`** — comment that suppresses errors on a specific line. Use only when the type checker is genuinely wrong; otherwise fix the underlying issue.

Treat these as flags, not solutions. Excessive `Any` and `# type: ignore` in a codebase usually means the types are wrong somewhere upstream.

In [ ]:
from typing import Any, cast

# Any — accepts anything, returns anything
def passthrough(x: Any) -> Any:
    return x

print(passthrough(42))
print(passthrough("hello"))

# cast — tell the checker, no runtime effect
import json
data: Any = json.loads('{"name": "ganesh", "age": 30}')
# After this point, the checker treats `user` as dict[str, str | int]
user = cast(dict[str, Any], data)
print(user["name"])

# type: ignore — last resort
result: int = "this is wrong but I'm sure" # type: ignore[assignment]

## `mypy` and `pyright` — the checkers

The runtime never checks annotations. A separate tool does:

- **`mypy`** — Dropbox's checker. The original. Strong, mature, sometimes slow on large codebases.
- **`pyright`** — Microsoft's checker. Fast (Rust-rewrite via `basedpyright`), what VS Code's Python extension uses under the hood. Stricter defaults than mypy.
- **`pyre`** — Meta's checker. Less common outside that orbit.
- **`ruff`** — primarily a linter and formatter, but ships with type-checking integration now.

Usage is the same in shape — `mypy src/`, `pyright src/`. Add to CI so type regressions block merge.

Config goes in `pyproject.toml`:

```toml
[tool.mypy]
strict = true
python_version = "3.12"

[tool.pyright]
typeCheckingMode = "strict"
pythonVersion = "3.12"
```

For new projects, **start strict**. Loosening later is harder than tightening from the beginning.

## Part 2 — Best practices

## PEP 8 and the tools that automate it

[PEP 8](https://peps.python.org/pep-0008/) is the official style guide for Python code. The high-confidence items: 4-space indentation, lowercase-with-underscores for functions and variables, CapWords for class names, lines under ~100 characters, 2 blank lines between top-level defs.

**You should not enforce PEP 8 by hand.** Two tools dominate:

- **`ruff format`** — formatter. Reformats your code to canonical style. Equivalent to (and faster than) `black`. Run it before every commit; you stop having opinions about whitespace.
- **`ruff check`** — linter. Catches dead code, unused imports, common bugs, style issues. Runs in milliseconds even on big codebases.

`ruff` is the Rust-based replacement for the old `black` + `isort` + `flake8` + `pylint` stack — does the same jobs, ten to a hundred times faster, configured once in `pyproject.toml`. New projects should default to `ruff`.

## Naming conventions

| Kind | Convention | Example |
|---|---|---|
| Module / package | `lowercase_with_underscores` | `my_module`, `data_pipeline` |
| Class | `CapWords` | `UserAccount`, `JSONParser` |
| Function, method, variable | `lowercase_with_underscores` | `find_user`, `total_count` |
| Constant | `UPPERCASE_WITH_UNDERSCORES` | `MAX_RETRIES`, `DEFAULT_TIMEOUT` |
| Internal-use | `_leading_underscore` | `_helper`, `_internal_state` |
| Name-mangled | `__double_leading` | `__secret` (rare; mostly for class internals) |
| Dunder | `__double_both__` | `__init__`, `__repr__` (reserved for Python) |

The most common violation: TitleCase methods because the writer came from Java. Run `ruff` once and it becomes a non-issue.

## Import sorting and grouping

PEP 8 groups imports into three blocks, separated by a blank line:

1. Standard library imports.
2. Related third-party imports.
3. Local application imports.

Within each block, sort alphabetically.

```python
import os
import sys
from pathlib import Path

import pandas as pd
import requests

from myproject.core import compute
from myproject.utils import slugify
```

`ruff check --select I` (or `isort` standalone) handles this automatically.

## Project layout — the `src/` layout

The recommended modern layout:

```text
   myproject/
     pyproject.toml
     README.md
     .gitignore
     .python-version
     src/
       myproject/
         __init__.py
         core.py
         utils/
           __init__.py
           text.py
     tests/
       test_core.py
       test_utils.py
```

The **src layout** (package nested inside `src/`) avoids a class of import bugs where your tests accidentally import from the working-directory `myproject/` instead of the installed one. The cost is one extra directory level; the payoff is reliable test-vs-installed isolation.

After `pip install -e .`, `from myproject.core import x` works from anywhere — your tests, your scripts, the REPL.

Tests live in their own `tests/` directory at the project root, not nested inside `src/myproject/`.

## `__init__.py` patterns

`__init__.py` runs when its package is first imported. Three common patterns:

- **Empty** — the file just marks the directory as a package. Most subpackages.
- **Re-exports** — `from .core import x, y, z` so users can write `from myproject import x` instead of `from myproject.core import x`. Common in the package root.
- **`__all__`** — a list of names exported by `from package import *`. Documentation, mostly; the modern bias is to not use `*` imports at all.

Keep `__init__.py` files small. Big logic in `__init__.py` runs on every import and is hard to debug.

```python
# myproject/__init__.py
from .core import compute, normalize
from .utils.text import slugify

__version__ = "0.1.0"
__all__ = ["compute", "normalize", "slugify"]
```

## Logging — `print` vs `logging`

`print()` is fine for scripts and notebooks. For anything that runs as a service, a job, or a library, use the **`logging`** module instead:

- **Logging respects levels.** `DEBUG`, `INFO`, `WARNING`, `ERROR`, `CRITICAL`. Filter by level in production.
- **Logging includes timestamp, module, line.** `print` doesn't.
- **Logging respects configuration.** One config in your entry point routes log lines to stderr, a file, a structured-log shipper — without changing the call sites.
- **Logging is composable across libraries.** A library that does `logger = logging.getLogger(__name__)` and logs to it lets the app decide where the logs go.

In [ ]:
import logging

# Configure once, at the top of your application
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
)

# Get a logger named after the current module — convention
logger = logging.getLogger(__name__)

logger.debug("won't show — below INFO level")
logger.info("user logged in")
logger.warning("retrying after timeout")
logger.error("failed to connect")

# Don't pre-format the message; let logging do it (avoids work when filtered out)
user = "ganesh"
n = 5
logger.info("user=%s attempted %d times", user, n)

## Configuration — environment variables, `.env` for development

Twelve-factor app guidance: read configuration from environment variables, not from hardcoded constants or checked-in config files.

For local development, **`python-dotenv`** loads a `.env` file (gitignored) into the environment. In production, real env vars come from the orchestrator (Kubernetes secrets, Docker compose, ECS task definitions).

```bash
pip install python-dotenv
```

```python
import os
from dotenv import load_dotenv

load_dotenv()                              # reads .env if present

DATABASE_URL = os.environ["DATABASE_URL"]  # required — fail loud if missing
TIMEOUT      = int(os.getenv("TIMEOUT", "30"))   # optional with default
```

`os.environ["X"]` raises `KeyError` if missing — good for required config. `os.getenv("X", default)` returns `None` (or the default) if missing — for optional config.

## Performance — measure before you optimize

Three layers of performance work, in the order you should reach for them:

1. **Algorithm.** O(n^2) versus O(n log n) matters more than any micro-optimization. Don't reach for the next two until you've checked the asymptotic shape.
2. **Profiling.** `cProfile` (built-in) shows where time is actually spent. `py-spy` (third-party) does the same for production processes without instrumentation. Profile first; optimize second.
3. **Micro-benchmarks.** Once you've identified the hot path, use `timeit` (or its `%timeit` magic in IPython/Jupyter) to compare alternatives.

The wrong order — guessing what's slow, optimizing things that aren't on the critical path — wastes time and adds complexity.

When you've measured and know the bottleneck is genuinely CPU-bound pure Python: vectorize with NumPy, rewrite the hot loop in Cython or Rust, or move the algorithm into a library that releases the GIL. Notebook 09 covers the GIL implications.

In [ ]:
import timeit

# Compare two ways to sum squares
loop_time = timeit.timeit(
    """
total = 0
for i in range(1000):
    total += i * i
""",
    number=10_000,
)

genexpr_time = timeit.timeit(
    "sum(i * i for i in range(1000))",
    number=10_000,
)

list_time = timeit.timeit(
    "sum([i * i for i in range(1000)])",
    number=10_000,
)

print(f"loop:     {loop_time:.3f}s")
print(f"gen expr: {genexpr_time:.3f}s")
print(f"list:     {list_time:.3f}s")

## Style choices that age well

Opinions you'll thank yourself for in two years:

- **Small functions.** A function that fits on a screen is debuggable. A function that doesn't isn't. Split aggressively.
- **Explicit data.** Pass dictionaries, dataclasses, NamedTuples — not "the third positional argument is the count." Type hints make this concrete.
- **Pure when you can.** A function that doesn't mutate its inputs and doesn't read global state is testable and composable. Reach for mutation only when you've decided you need it.
- **Hard-to-misuse APIs.** Keyword-only flags, default arguments that fail loud when wrong, names that say what they mean (`timeout_seconds` not `timeout`). The reader of your code in six months is yourself with no memory of the conversation that led to this design.
- **Comments explain *why*, not *what*.** The code says what. The comment says why it's *that way* and not the other way.
- **Tests as the second documentation.** A failing test pins down what was promised. A passing test demonstrates how the API is meant to be used. Notebook 10 covers `pytest`.

## What's next

You now have the typing toolkit — type hints, the static checkers, the structural typing protocol — and the broader practices that make a Python codebase healthy: style automation, project layout, logging, configuration, the "measure before you optimize" rule.

Notebook 09 covers concurrency — the GIL, threads, processes, and the asyncio coroutine story. The runtime model that decides what you can and cannot speed up by running things in parallel.